> **Drive MCP upload note:** Self-contained 1.62MB notebook (embedded slide images) lives on the box at `/workspace/cml_sessions/s7_cv_reg/out/CML7_Cross_Validation_Regularization.ipynb`. This Drive file is the executable teaching notebook (all demo code + quizzes). Re-run cells for plots.


# Cross Validation and Regularization

**Scaler CML | Applied ML (Intro)**
**Instructor:** Chitwan Manchanda
**Session:** CML Group B | Session 7


# 1. Agenda
1. Regularization (Ridge/Lasso/ElasticNet)
2. Hyperparameters vs parameters
3. Holdout + K-fold CV
4. RidgeCV/LassoCV
5. Pitfalls


# Quiz 1 — role of λ? optimal trade-off


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
np.random.seed(42)
data = datasets.load_diabetes()
X, y = data['data'], data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
degree = 5
poly = PolynomialFeatures(degree=degree, include_bias=False)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(poly.fit_transform(X_train))
X_test_s = scaler.transform(poly.transform(X_test))
print('Poly width', X_train_s.shape[1])
ols = LinearRegression().fit(X_train_s, y_train)
print('OLS train R2', round(ols.score(X_train_s, y_train), 4), 'test R2', round(ols.score(X_test_s, y_test), 4))


In [ ]:
def report(name, model):
    model.fit(X_train_s, y_train)
    tr, te = model.score(X_train_s, y_train), model.score(X_test_s, y_test)
    nnz = int(np.sum(np.abs(getattr(model, 'coef_', np.array([]))) > 1e-8))
    print(f'{name:28s} train={tr:7.3f} test={te:7.3f} nnz={nnz}')
report('Lasso a=0.01', Lasso(alpha=0.01, max_iter=8000, tol=1e-3))
report('Ridge a=0.01', Ridge(alpha=0.01))
report('Lasso a=1e6', Lasso(alpha=1e6, max_iter=5000))
report('Ridge a=1e6', Ridge(alpha=1e6))
report('Lasso a=1', Lasso(alpha=1.0, max_iter=8000, tol=1e-3))
report('Ridge a=100', Ridge(alpha=100))
report('ElasticNet', ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=8000, tol=1e-3))


In [ ]:
alphas = np.logspace(-2, 3, 12)
lasso_nnz, lasso_test, ridge_test = [], [], []
for a in alphas:
    las = Lasso(alpha=a, max_iter=6000, tol=1e-2).fit(X_train_s, y_train)
    rid = Ridge(alpha=a).fit(X_train_s, y_train)
    lasso_nnz.append(np.sum(np.abs(las.coef_) > 1e-8))
    lasso_test.append(las.score(X_test_s, y_test))
    ridge_test.append(rid.score(X_test_s, y_test))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(alphas, lasso_nnz, marker='o'); axes[0].set_xscale('log'); axes[0].set_title('Lasso sparsity')
axes[1].plot(alphas, lasso_test, label='Lasso'); axes[1].plot(alphas, ridge_test, label='Ridge')
axes[1].set_xscale('log'); axes[1].legend(); axes[1].set_title('Test R2 vs alpha')
plt.tight_layout(); plt.show()


# Quizzes 2–4: ElasticNet mix; λ=0 → overfit; pick best λ on validation not test


In [ ]:
np.random.seed(2)
X = np.random.rand(1000, 1)
y = 0.7*(X**5)-2.1*(X**4)+2.3*(X**3)+0.2*(X**2)+0.3*X+0.4*np.random.rand(1000,1)
def adj_r2(X_mat, y_true, r2):
    n, p = len(y_true), X_mat.shape[1]
    denom = n - p - 1
    return np.nan if denom <= 0 else 1 - (1 - r2) * (n - 1) / denom
X_tr_cv, X_test, y_tr_cv, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
X_train, X_val, y_train, y_val = train_test_split(X_tr_cv, y_tr_cv, test_size=0.25, random_state=1)
print('shapes', X_train.shape, X_val.shape, X_test.shape)
max_degree = 11
train_scores, val_scores = [], []
for degree in range(1, max_degree):
    pipe = make_pipeline(PolynomialFeatures(degree), StandardScaler(), Ridge())
    pipe.fit(X_train, y_train)
    Xt = pipe.named_steps['polynomialfeatures'].transform(X_train)
    Xv = pipe.named_steps['polynomialfeatures'].transform(X_val)
    train_scores.append(adj_r2(Xt, y_train, pipe.score(X_train, y_train)))
    val_scores.append(adj_r2(Xv, y_val, pipe.score(X_val, y_val)))
print('Best degree by validation', int(np.argmax(val_scores))+1)
plt.plot(range(1,max_degree), train_scores, label='train'); plt.plot(range(1,max_degree), val_scores, label='val')
plt.legend(); plt.show()
degree_star = 3
rate_list = [0.01, 0.1, 1, 5, 10]
train_scores, val_scores = [], []
for rate in rate_list:
    pipe = make_pipeline(PolynomialFeatures(degree_star), StandardScaler(), Ridge(alpha=rate))
    pipe.fit(X_train, y_train)
    Xt = pipe.named_steps['polynomialfeatures'].transform(X_train)
    Xv = pipe.named_steps['polynomialfeatures'].transform(X_val)
    train_scores.append(adj_r2(Xt, y_train, pipe.score(X_train, y_train)))
    val_scores.append(adj_r2(Xv, y_val, pipe.score(X_val, y_val)))
best_rate = rate_list[int(np.argmax(val_scores))]
print('Best lambda by validation', best_rate)
final = make_pipeline(PolynomialFeatures(degree_star), StandardScaler(), Ridge(alpha=best_rate))
final.fit(X_train, y_train)
for name, Xx, yy in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    Xm = final.named_steps['polynomialfeatures'].transform(Xx)
    print(name, adj_r2(Xm, yy, final.score(Xx, yy)))


In [ ]:
np.random.seed(2)
X_small = np.random.rand(100, 1)
y_small = 0.7*(X_small**5)-2.1*(X_small**4)+2.3*(X_small**3)+0.2*(X_small**2)+0.3*X_small+0.4*np.random.rand(100,1)
kf = KFold(n_splits=10, shuffle=True, random_state=2)
train_scores, val_scores = [], []
for degree in range(1, 8):
    fold_train, fold_val = [], []
    for tr, va in kf.split(X_small):
        pipe = make_pipeline(PolynomialFeatures(degree), StandardScaler(), LinearRegression())
        pipe.fit(X_small[tr], y_small[tr])
        Xt = pipe.named_steps['polynomialfeatures'].transform(X_small[tr])
        Xv = pipe.named_steps['polynomialfeatures'].transform(X_small[va])
        fold_train.append(adj_r2(Xt, y_small[tr], pipe.score(X_small[tr], y_small[tr])))
        fold_val.append(adj_r2(Xv, y_small[va], pipe.score(X_small[va], y_small[va])))
    train_scores.append(np.nanmean(fold_train)); val_scores.append(np.nanmean(fold_val))
print('Best degree K-fold', int(np.argmax(val_scores))+1)
plt.plot(range(1,8), train_scores); plt.plot(range(1,8), val_scores); plt.show()


In [ ]:
Xd, yd = datasets.load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(Xd, yd, test_size=0.2, random_state=0)
alphas = np.logspace(-3, 3, 30)
ridge_cv = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5)).fit(X_tr, y_tr)
print('RidgeCV alpha', ridge_cv.named_steps['ridgecv'].alpha_, 'test R2', round(ridge_cv.score(X_te, y_te), 4))
lasso_cv = make_pipeline(StandardScaler(), LassoCV(alphas=alphas, cv=5, max_iter=8000, random_state=0)).fit(X_tr, y_tr)
print('LassoCV alpha', round(lasso_cv.named_steps['lassocv'].alpha_, 5), 'test R2', round(lasso_cv.score(X_te, y_te), 4))
scores = cross_val_score(make_pipeline(StandardScaler(), Ridge(alpha=1.0)), X_tr, y_tr, cv=5, scoring='r2')
print('5-fold Ridge a=1', round(scores.mean(),4), '+/-', round(scores.std(),4))


# Pitfalls
1. Do not tune on the final test set.
2. Scale before Ridge/Lasso/ElasticNet.
3. Fit transformers on train only (Pipeline / per fold).
4. Huge alpha underfits.

# Wrap-up
Ridge shrinks; Lasso sparsifies; ElasticNet mixes. λ is a hyperparameter — choose with validation/CV, report test once.
